In [1]:
import numpy as np
import pandas as pd
import random

In [ ]:
# Some hand-crafted rules to avoid double counting of issues in Bandit and CodeQL reports.
same_rules = [
    ("B103:set_bad_file_permissions","Overly permissive file permissions"),
    ('B301:blacklist','Deserialization of user-controlled data'),
    ('B302:blacklist', 'Deserialization of user-controlled data'),
    ('B403:blacklist', 'Deserialization of user-controlled data'),
    ('B608:hardcoded_sql_expressions','SQL query built from user-controlled sources'),
    ('B505:weak_cryptographic_key','Use of weak cryptographic key'),
    ('B501:request_with_no_cert_validation','Request without certificate validation'),
    ('B507:ssh_no_host_key_verification','Accepting unknown SSH host keys when using Paramiko'),
    ('B108:hardcoded_tmp_directory','Insecure temporary file'),
    ('B306:blacklist','Insecure temporary file'),
    ('B324:hashlib', 'Use of a broken or weak cryptographic algorithm'),
    ('B502:ssl_with_bad_version','Use of insecure SSL/TLS version'),
    ('B503:ssl_with_bad_defaults','Use of insecure SSL/TLS version'),
    ('B504:ssl_with_no_version','Default version of SSL/TLS may be insecure'),
    ('B303:blacklist', 'Use of a broken or weak cryptographic algorithm'),
    ('B304:blacklist', 'Use of a broken or weak cryptographic algorithm'),
    ('B305:blacklist', 'Use of a broken or weak cryptographic algorithm'),
    ('B102:exec_used', 'Uncontrolled command line'),
    ('B601:paramiko_calls', 'Uncontrolled command line'),
    ('B602:subprocess_popen_with_shell_equals_true', 'Uncontrolled command line'),
    ('B603:subprocess_without_shell_equals_true', 'Uncontrolled command line'),
    ('B604:any_other_function_with_shell_equals_true', 'Uncontrolled command line'),
    ('B605:start_process_with_a_shell', 'Uncontrolled command line'),
    ('B606:start_process_with_no_shell', 'Uncontrolled command line'),
    ('B607:start_process_with_partial_path', 'Uncontrolled command line'),
    ('B609:linux_commands_wildcard_injection', 'Uncontrolled command line'),
    ('B307:blacklist', 'Uncontrolled command line'),
    ('B102:exec_used', 'Unsafe shell command constructed from library input'),
    ('B601:paramiko_calls', 'Unsafe shell command constructed from library input'),
    ('B602:subprocess_popen_with_shell_equals_true', 'Unsafe shell command constructed from library input'),
    ('B603:subprocess_without_shell_equals_true', 'Unsafe shell command constructed from library input'),
    ('B604:any_other_function_with_shell_equals_true', 'Unsafe shell command constructed from library input'),
    ('B605:start_process_with_a_shell', 'Unsafe shell command constructed from library input'),
    ('B606:start_process_with_no_shell', 'Unsafe shell command constructed from library input'),
    ('B607:start_process_with_partial_path', 'Unsafe shell command constructed from library input'),
    ('B609:linux_commands_wildcard_injection', 'Unsafe shell command constructed from library input'),
    ('B307:blacklist', 'Unsafe shell command constructed from library input') 
]

bandit_same_rules = {i[0]:True for i in same_rules}
codeql_same_rules = {i[1]:True for i in same_rules}

In [ ]:
#Functions to extract information from the reports and calculate metrics

def extract_number(array,phrase):
    subset = [i  for i in array if phrase in i ]
    if(phrase=="Files skipped "):
        return int(subset[0].split(" ")[-1][1:-2])
    else:
        return int(subset[0].split(" ")[-1])
    
def find_number_of_buggy_files_in_report(test_data_length,report):
    possible_files = [f"code_{i}.py" for i in range(0,len(test_data_length))]
    list_of_buggy_files = []
    for i in possible_files:
        if i in report:
            list_of_buggy_files.append(i)
    list_of_buggy_files = set(list_of_buggy_files)
    return list_of_buggy_files

def get_bandit_issues(path_to_bandit_report):
    f = open(path_to_bandit_report,"r")
    whole = f.read()
    lines = whole.splitlines()

    #Extract the number of skipped files
    skipped_files = extract_number(lines,"Files skipped ")

    #Extract the issues portion
    issues = whole[whole.find("Test results:")+len("Test results:"):whole.find("Code scanned:")-1].strip()
    issues = issues.split("--------------------------------------------------")
    issues = [i for i in issues if len(i.strip())>0]
    issues = [i.strip() for i in issues]

    return issues,skipped_files

def get_codeql_issues(path_to_codeql_report):
    f = open(path_to_codeql_report,"r")
    lines = f.read().splitlines()
    return lines 

def get_buggy_file_metric_info(bandit_issues,codeql_issues,length_of_dataset):
    buggy_file_ids = {}
    for i in range(0,length_of_dataset):
        filename = f"code_{i}.py"
        for j in bandit_issues:
            if filename in j:
                buggy_file_ids[filename]=True
        for j in codeql_issues:
            if filename in j:
                buggy_file_ids[filename]=True
    
    return buggy_file_ids,len(buggy_file_ids)

def number_of_repeated_rules(relevant_bandit_issues,relevant_codeql_issues):
    number_of_repeats = 0
    relevant_bandit_same_rules = []
    relevant_codeql_same_rules = []

    for item in bandit_same_rules:
        for issue in relevant_bandit_issues:
            if item in issue:
                relevant_bandit_same_rules.append(item)
    
    # print(relevant_codeql_issues)
    for item in codeql_same_rules:
        for issue in relevant_codeql_issues:
            if item in issue:
                relevant_codeql_same_rules.append(item)
    
    # print(relevant_bandit_same_rules)
    # print(relevant_codeql_same_rules)
    
    for item in relevant_bandit_same_rules:
        conflicting_codeql_issues = [i[1] for i in same_rules if i[0]==item]
        for idx,codeql_item in enumerate(relevant_codeql_same_rules):
            if(codeql_item in conflicting_codeql_issues):
                number_of_repeats+=1
                relevant_codeql_same_rules[idx] = "#"

    # print(number_of_repeats)
    # print("-------")      
    
    return number_of_repeats


def get_number_of_bugs(bandit_issues,codeql_issues,length_of_dataset):
    number_of_bugs = 0

    for i in range(0,length_of_dataset):
        # print(i)
        filename = f"code_{i}.py"
        relevant_bandit_issues = []
        relevant_codeql_issues = []

        for j in bandit_issues:
            if filename in j:
                relevant_bandit_issues.append(j)

        for j in codeql_issues:
            if filename in j:
                relevant_codeql_issues.append(j)
        
        temp_number_of_bugs = len(relevant_bandit_issues)+len(relevant_codeql_issues)
        # temp_number_of_bugs = len(relevant_codeql_issues)
        #temp_number_of_bugs = len(relevant_bandit_issues)

        number_of_repeats = number_of_repeated_rules(relevant_bandit_issues,relevant_codeql_issues)
        #number_of_repeats = 0
        
        number_of_bugs+=temp_number_of_bugs-number_of_repeats

    return number_of_bugs
        


def security_analysis(path_to_report,length_of_dataset):

    # bandit_issues,number_of_skipped_files = get_bandit_issues(f"{path_to_report}/code_3.txt")
    # codeql_issues = get_codeql_issues(f"{path_to_report}/codeql_analysis_copy.csv")
    bandit_issues,number_of_skipped_files = get_bandit_issues(f"{path_to_report}/bandit_analysis.txt")
    codeql_issues = get_codeql_issues(f"{path_to_report}/codeql_analysis.csv")

    file_metric = get_buggy_file_metric_info(bandit_issues,codeql_issues,len_dataset)
    bug_metric = get_number_of_bugs(bandit_issues,codeql_issues,len_dataset)
    

    f = 100*(file_metric[1])/(length_of_dataset-number_of_skipped_files)
    # f = 100-f
    b = 100*bug_metric/(length_of_dataset-number_of_skipped_files)
    skip = 100*number_of_skipped_files/length_of_dataset

    #print(f"{f:.1f} & {b:.2f} & {skip:.2f}")
    print(f"{f:.1f} & {b:.0f}")
    #print(f"{b:.2f}")

    return file_metric,bug_metric

In [ ]:
# Enter the information here to calculate the metrics
filepath = "Enter the path to the report directory here"
len_dataset = "Enter the length of the dataset here"
NO_OF_PARSES = 5

security_analysis(filepath,len_dataset)